In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run utilityfilepath

In [0]:
dbutils.widgets.text("catalog", "ecommerce", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

In [0]:
catalog= dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path=f'datasetpath{data_source}'
print(base_path)

In [0]:
df_bronze= spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

In [0]:
df_dropdupe= df_bronze.dropDuplicates(['customer_id'])
df_dropdupe.count()

In [0]:
df_clean = df_dropdupe.withColumn(
    "customer_city",
    F.initcap(F.col("customer_city"))
)

In [0]:
df_silver=df_clean


In [0]:
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
df_silver= spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")

df_gold=df_silver.select('customer_id','customer_city','customer_state')


In [0]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")